# PE-SC Project Part 1 (fixed)

Fixes applied vs. the original notebook (see chat / README for the full list):
- The Implication model now sees **both** X1 and X2, not just X1.
- Evaluation metrics are computed on the model's actual predictions, not the scaled X test values.
- Separate `MinMaxScaler` instances are used for X and y instead of one shared, repeatedly-refit scaler.
- Added classification-style metrics (accuracy/F1) alongside RMSE/R2, since `Target` is actually binary.
- The five near-identical "smallest two rows" cells are now one small function.
- Paths no longer assume Google Colab's `/content/...` — the notebook looks for the `.xlsx` files next to itself.

In [ ]:
import os
import zipfile

# Works both in Colab (if you upload archive.zip) and when run from a clone of the
# repo, where the .xlsx files already sit next to this notebook.
DATA_DIR = "."
zip_file_path = "archive.zip"

if os.path.exists(zip_file_path):
    with zipfile.ZipFile(zip_file_path, "r") as zip_ref:
        zip_ref.extractall(DATA_DIR)

print(f"Using data directory: {os.path.abspath(DATA_DIR)}")

In [ ]:
extracted_files = [f for f in os.listdir(DATA_DIR) if f.lower().endswith(".xlsx")]
print(extracted_files)

In [ ]:
!pip install openpyxl

In [ ]:
import pandas as pd

file_path = os.path.join(DATA_DIR, "Implication Fuzzy.xlsx")
df = pd.read_excel(file_path)

df.head(2)

In [ ]:
df1 = pd.read_excel(os.path.join(DATA_DIR, "XOR Fuzzy.xlsx"))
df1.head(2)

In [ ]:
df2 = pd.read_excel(os.path.join(DATA_DIR, "OR Fuzzy.xlsx"))
df2.head(2)

In [ ]:
df3 = pd.read_excel(os.path.join(DATA_DIR, "AND Fuzzy.xlsx"))
df3.head(2)

In [ ]:
df4 = pd.read_excel(os.path.join(DATA_DIR, "NOT Fuzzy.xlsx"))
df4.head(2)

In [ ]:
print(f"ROWS: {df.shape[0]}")
print(f"COLUMNS: {df.shape[1]}")

In [ ]:
df.isnull().sum()

In [ ]:
df["Target"].value_counts()

In [ ]:
import matplotlib.pyplot as plt
plt.scatter(x=df['Target'], y=df['X1'], color='g')
plt.title("X1 vs. Target");

In [ ]:
plt.scatter(x=df['X2'], y=df['Target'], color='r')
plt.title("X2 vs. Target");

## Implication model

**Fix:** the original only fed the model `X1`. Implication is a function of *two* inputs (`X1`, `X2`), so it could never learn the real relationship with one input missing. Separate scalers are used for X and y so fitting one never overwrites the other's learned range.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

x_scaler = MinMaxScaler()
y_scaler = MinMaxScaler()

X = df[["X1", "X2"]].values          # both inputs of the Implication function
targ = df["Target"].values.reshape(-1, 1)

sx = x_scaler.fit_transform(X)
sy = y_scaler.fit_transform(targ)

scaled_x_train, scaled_x_test, scaled_y_train, scaled_y_test = train_test_split(
    sx, sy, test_size=0.20, random_state=13
)
scaled_x_train.shape, scaled_x_test.shape

In [ ]:
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import GridSearchCV

params = {
    "hidden_layer_sizes": [(3, 3), (3,), (3, 3, 3)],
    "activation": ["logistic", "relu", "tanh"],
    "solver": ["sgd", "lbfgs", "adam"],
    "verbose": [0],
    "max_iter": [10000],
}

mlp_grid = GridSearchCV(estimator=MLPRegressor(), param_grid=params)
mlp_grid.fit(scaled_x_train, scaled_y_train.ravel())
mlp_grid

In [ ]:
# Fix: inverse-transform the model's actual predictions (y_predict), not the
# scaled X test values, and use the y-specific scaler.
y_predict = mlp_grid.predict(scaled_x_test)
y_predict_orig = y_scaler.inverse_transform(y_predict.reshape(-1, 1))
y_test_orig = y_scaler.inverse_transform(scaled_y_test)

In [ ]:
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from math import sqrt

RMSE = float(format(sqrt(mean_squared_error(y_test_orig, y_predict_orig)), '.3f'))
MSE = mean_squared_error(y_test_orig, y_predict_orig)
MAE = mean_absolute_error(y_test_orig, y_predict_orig)
r2 = r2_score(y_test_orig, y_predict_orig)

print(f"RMSE = {RMSE}\nMSE = {MSE}\nMAE = {MAE}\nR2 = {r2}")

In [ ]:
# Target is actually binary (0/1), so it's also worth scoring this as classification,
# not just regression -- threshold the (0-1) predictions at 0.5.
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

y_pred_class = (y_predict_orig.ravel() >= 0.5).astype(int)
y_true_class = y_test_orig.ravel().astype(int)

accuracy = accuracy_score(y_true_class, y_pred_class)
f1 = f1_score(y_true_class, y_pred_class)

print(f"Accuracy = {accuracy:.3f}\nF1 = {f1:.3f}")
print("Confusion matrix:")
print(confusion_matrix(y_true_class, y_pred_class))

## Smallest-two-rows lookup, deduplicated

**Fix:** cells 17-21 of the original repeated the same "find the 2 rows with the smallest min(X1, X2)" logic five times, once per file, changing only the filename. That's now one function.

In [ ]:
def smallest_two(path, cols=("X1", "X2")):
    """Return the 2 rows of `path` with the smallest value across `cols`
    (row-wise min when there are 2+ columns, or the plain column when there's 1)."""
    data = pd.read_excel(path)
    if len(cols) == 1:
        return data.nsmallest(2, cols[0])
    data = data.copy()
    data["min_value"] = data[list(cols)].min(axis=1)
    return data.nsmallest(2, "min_value").drop(columns=["min_value"])


print(smallest_two(os.path.join(DATA_DIR, "Implication Fuzzy.xlsx")))
print(smallest_two(os.path.join(DATA_DIR, "AND Fuzzy.xlsx")))
print(smallest_two(os.path.join(DATA_DIR, "OR Fuzzy.xlsx")))
print(smallest_two(os.path.join(DATA_DIR, "NOT Fuzzy.xlsx"), cols=("X1",)))
print(smallest_two(os.path.join(DATA_DIR, "XOR Fuzzy.xlsx")))

## XOR model

Same shape as the Implication model above, but this one already used both `X1` and `X2` in the original notebook. It's rewritten here with the same separate-scaler fix and clearer naming (`df_xor` instead of the reused, ambiguous `df`).

In [ ]:
import numpy as np
from sklearn.model_selection import GridSearchCV
from sklearn.neural_network import MLPRegressor

df_xor = pd.read_excel(os.path.join(DATA_DIR, "XOR Fuzzy.xlsx"))

x_scaler = MinMaxScaler()
y_scaler = MinMaxScaler()

X = df_xor[["X1", "X2"]].values
y = df_xor["Target"].values.reshape(-1, 1)

X_scaled = x_scaler.fit_transform(X)
y_scaled = y_scaler.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_scaled, test_size=0.2, random_state=13)

param_grid = {
    "hidden_layer_sizes": [(3, 3), (3,), (3, 3, 3)],
    "activation": ["logistic", "relu", "tanh"],
    "solver": ["sgd", "lbfgs", "adam"],
    "max_iter": [10000],
    "verbose": [0],
}

mlp_grid = GridSearchCV(estimator=MLPRegressor(), param_grid=param_grid, scoring="neg_mean_squared_error", cv=5)
mlp_grid.fit(X_train, y_train.ravel())

y_pred_train = mlp_grid.predict(X_train)
y_pred_test = mlp_grid.predict(X_test)

y_train_orig = y_scaler.inverse_transform(y_train)
y_test_orig = y_scaler.inverse_transform(y_test)
y_pred_train_orig = y_scaler.inverse_transform(y_pred_train.reshape(-1, 1))
y_pred_test_orig = y_scaler.inverse_transform(y_pred_test.reshape(-1, 1))

rmse_train = sqrt(mean_squared_error(y_train_orig, y_pred_train_orig))
rmse_test = sqrt(mean_squared_error(y_test_orig, y_pred_test_orig))
mse_train = mean_squared_error(y_train_orig, y_pred_train_orig)
mse_test = mean_squared_error(y_test_orig, y_pred_test_orig)
mae_train = mean_absolute_error(y_train_orig, y_pred_train_orig)
mae_test = mean_absolute_error(y_test_orig, y_pred_test_orig)
r2_train = r2_score(y_train_orig, y_pred_train_orig)
r2_test = r2_score(y_test_orig, y_pred_test_orig)

print(f"Train RMSE: {rmse_train}, Test RMSE: {rmse_test}")
print(f"Train MSE: {mse_train}, Test MSE: {mse_test}")
print(f"Train MAE: {mae_train}, Test MAE: {mae_test}")
print(f"Train R2: {r2_train}, Test R2: {r2_test}")

In [ ]:
param_grid = {
    "hidden_layer_sizes": [(5, 5), (5,), (5, 5, 5)],
    "activation": ["logistic", "relu", "tanh"],
    "solver": ["sgd", "lbfgs", "adam"],
    "max_iter": [10000],
    "verbose": [0],
}

mlp_grid = GridSearchCV(estimator=MLPRegressor(), param_grid=param_grid, scoring="neg_mean_squared_error", cv=5)
mlp_grid.fit(X_train, y_train.ravel())

y_pred_train = mlp_grid.predict(X_train)
y_pred_test = mlp_grid.predict(X_test)

y_pred_train_orig = y_scaler.inverse_transform(y_pred_train.reshape(-1, 1)).flatten()
y_pred_test_orig = y_scaler.inverse_transform(y_pred_test.reshape(-1, 1)).flatten()

rmse_train = sqrt(mean_squared_error(y_train_orig, y_pred_train_orig))
rmse_test = sqrt(mean_squared_error(y_test_orig, y_pred_test_orig))

print(f"Updated Train RMSE: {rmse_train}, Updated Test RMSE: {rmse_test}")